In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.9145000000000001, 10: 0.9075000000000001, 20: 0.9105000000000004, 30: 0.9075, 40: 0.9065, 50: 0.9115000000000002, 60: 0.9055000000000003, 70: 0.9115000000000002, 80: 0.9105000000000002, 90: 0.9055000000000002, 100: 0.9155000000000003, 110: 0.9135, 120: 0.9095000000000002, 130: 0.9125000000000003, 140: 0.9125, 150: 0.9105000000000001, 160: 0.9110000000000001, 170: 0.9130000000000003, 180: 0.9195000000000002, 190: 0.9110000000000001, 200: 0.9130000000000003, 210: 0.9175000000000002, 220: 0.917, 230: 0.9120000000000003, 240: 0.913, 250: 0.9205000000000002, 260: 0.9179999999999996, 270: 0.916, 280: 0.9205, 290: 0.9145000000000003, 300: 0.9155000000000001}
{0: 0.0017597499999999998, 10: 0.0019537499999999998, 20: 0.0013997499999999997, 30: 0.00185375, 40: 0.00196775, 50: 0.0017577500000000002, 60: 0.0015397499999999997, 70: 0.0014177499999999995, 80: 0.0013597499999999994, 90: 0.0018797499999999993, 100: 0.0017897499999999997, 110: 0.0013277499999999997, 120: 0.0018597499999999999, 13